# Strict CV Fold Design

This notebook designs stricter validation folds after the analog-gated Huber + Hilbert model improved a bounded local residual sample but worsened public LB. The goal is to make future local evaluation stress far-analog wells, descriptor clusters, and high-extrapolation wells before LightGBM/XGBoost or deeper Bayesian models are tuned.


## Design Targets

The folds must stay well-level, be deterministic, avoid using target-zone labels for assignment, and export reusable fold tables for later notebooks. We score each candidate fold design by balance, fold-safe nearest-training analog distance, and whether it isolates high-extrapolation or weak-analog wells.


In [1]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent.parent.parent

RUN_ID = "2026-06-26__strict_cv_fold_design"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
for path in [RUN_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "n_folds": 3,
    "n_descriptor_clusters": 9,
    "analog_candidate": "log_plus_trajectory_k1_progress_gate",
    "recommended_fold_column": "strict_hybrid_fold",
    "random_state": 20260626,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

"Set up deterministic fold-design configuration."
print(ROOT)
print(RUN_DIR)


C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-26__strict_cv_fold_design


## Well Descriptor Assembly


In [2]:
descriptor_path = TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv"
selection_path = TABLE_DIR / "model__analog_selection_refinement_selection_diagnostics__2026-06-23.csv"
well_diag_path = TABLE_DIR / "model__alignment_hsgp_residual_well_diagnostics__2026-06-23.csv"

descriptors = pd.read_csv(descriptor_path)
selection = pd.read_csv(selection_path)
well_diag = pd.read_csv(well_diag_path)

best_selection = selection.loc[selection["candidate"].eq(CONFIG["analog_candidate"])].copy()
best_selection = best_selection[["fold", "wellname", "selected_analogs", "mean_distance"]].rename(columns={"mean_distance": "selected_analog_distance"})

raw_rows = []
train_dir = ROOT / "data" / "raw" / "train"
for csv_path in sorted(train_dir.glob("*__horizontal_well.csv")):
    wellname = csv_path.name.split("__")[0]
    df = pd.read_csv(csv_path, usecols=lambda col: col in {"MD", "TVT", "TVT_input", "GR", "X", "Y", "Z"})
    known_mask = df["TVT_input"].notna()
    target_mask = ~known_mask
    known = df.loc[known_mask]
    target = df.loc[target_mask]
    raw_rows.append({
        "wellname": wellname,
        "total_rows": int(len(df)),
        "known_rows_raw": int(known_mask.sum()),
        "target_rows": int(target_mask.sum()),
        "target_row_share": float(target_mask.mean()),
        "known_tvt_min": float(known["TVT"].min()) if len(known) else np.nan,
        "known_tvt_max": float(known["TVT"].max()) if len(known) else np.nan,
        "target_tvt_min": float(target["TVT"].min()) if len(target) else np.nan,
        "target_tvt_max": float(target["TVT"].max()) if len(target) else np.nan,
        "target_tvt_span": float(target["TVT"].max() - target["TVT"].min()) if len(target) else 0.0,
        "target_gr_missing_share": float(target["GR"].isna().mean()) if len(target) else np.nan,
    })
raw_summary = pd.DataFrame(raw_rows)
raw_summary["target_start_gap"] = raw_summary["target_tvt_min"] - raw_summary["known_tvt_max"]

well_table = descriptors.merge(raw_summary, on="wellname", how="left")
well_table = well_table.merge(best_selection, on="wellname", how="left")
well_table = well_table.merge(well_diag[["wellname", "mean_residual_before", "mean_residual_after", "abs_bias_delta"]], on="wellname", how="left")
well_table = well_table.drop_duplicates(subset=["wellname"]).reset_index(drop=True)

stress_cols = ["target_rows", "target_row_share", "target_tvt_span", "target_start_gap", "selected_analog_distance", "known_tvt_span", "xy_span_known", "z_drop_known"]
for col in stress_cols:
    well_table[col + "_rank_pct"] = well_table[col].rank(pct=True, method="average")
well_table["extrapolation_stress_score"] = well_table[["target_rows_rank_pct", "target_row_share_rank_pct", "target_tvt_span_rank_pct", "target_start_gap_rank_pct"]].mean(axis=1)
well_table["analog_stress_score"] = well_table["selected_analog_distance_rank_pct"]
well_table["combined_stress_score"] = 0.45 * well_table["extrapolation_stress_score"] + 0.35 * well_table["analog_stress_score"] + 0.20 * well_table["known_tvt_span_rank_pct"]

descriptor_export = TABLE_DIR / "validation__strict_cv_well_descriptors__2026-06-26.csv"
well_table.to_csv(descriptor_export, index=False)
shutil.copy2(descriptor_export, RUN_DIR / descriptor_export.name)

print(well_table.shape)
well_table.head()


(773, 43)


,wellname,known_rows,known_tvt_span,known_md_span,known_gr_mean_last300,known_gr_std_last300,known_gr_slope_last300,xy_span_known,z_drop_known,last_known_tvt,typewell_corr_last300,typewell_mae_last300,typewell_best_lag_corr,typewell_best_lag_tvt,typewell_best_lag_mae,total_rows,known_rows_raw,target_rows,target_row_share,known_tvt_min,known_tvt_max,target_tvt_min,target_tvt_max,target_tvt_span,target_gr_missing_share,target_start_gap,fold,selected_analogs,selected_analog_distance,mean_residual_before,mean_residual_after,abs_bias_delta,target_rows_rank_pct,target_row_share_rank_pct,target_tvt_span_rank_pct,target_start_gap_rank_pct,selected_analog_distance_rank_pct,known_tvt_span_rank_pct,xy_span_known_rank_pct,z_drop_known_rank_pct,extrapolation_stress_score,analog_stress_score,combined_stress_score
0,000d7d20,1442.0,520.10,1441.0,106.326237,6.068260,3.165573,1190.815381,-476.51,11747.37,-0.398691,11.532797,0.553606,-22.5,19.894136,5278,1442,3836,0.726790,11236.02,11756.12,11734.83,11750.03,15.20,0.473410,-21.29,1,ae0784a8,0.316157,-3.284996,-3.747827,0.462831,0.201811,0.401035,0.103493,0.366106,0.241915,0.143596,0.346701,0.910737,0.268111,0.241915,0.234039
1,00bbac68,1545.0,818.42,1544.0,87.895091,10.138084,6.181385,1077.617096,-736.48,12223.54,0.096803,10.819063,0.773944,-15.0,8.951110,7559,1545,6014,0.795608,11406.63,12225.05,12189.82,12240.01,50.19,0.138344,-35.23,1,940a48d9,0.416028,-9.038254,-8.242219,-0.796035,0.821475,0.869340,0.926261,0.125485,0.561449,0.685640,0.090556,0.481242,0.685640,0.561449,0.642173
2,00e12e8b,2083.0,998.64,2082.0,32.240742,6.211680,-0.113335,1453.191016,-968.90,11604.82,0.768869,8.657913,0.768869,0.0,8.657913,6384,2083,4301,0.673716,10606.18,11604.82,11587.05,11616.64,29.59,0.097187,-17.77,1,98177ced,0.578991,-6.779790,-6.520778,-0.259013,0.326649,0.147477,0.604140,0.438551,0.865459,0.958603,0.936611,0.042691,0.379204,0.865459,0.665273
3,015fe0d2,1654.0,493.93,1653.0,72.280501,16.718185,12.892282,1415.574133,-517.29,12233.78,0.855918,10.921105,0.876637,-40.0,7.860428,5950,1654,4296,0.722017,11747.26,12241.19,12232.92,12271.53,38.61,0.320298,-8.27,1,b37fd114,0.728474,11.398129,10.686890,-0.711240,0.322122,0.366106,0.803364,0.752264,0.957309,0.103493,0.910737,0.860285,0.560964,0.957309,0.608190
4,01869cd4,1293.0,387.67,1292.0,108.910689,30.002244,3.904767,1103.679180,-468.21,11745.94,0.895327,14.958429,0.895327,0.0,14.958429,6850,1293,5557,0.811241,11358.27,11745.94,11728.75,11767.75,39.00,0.511247,-17.19,1,9ad5410a,0.803998,6.890234,6.981438,0.091204,0.710220,0.931436,0.809832,0.456016,0.978008,0.023286,0.131953,0.917206,0.726876,0.978008,0.674054


## Candidate Fold Builders


In [3]:
feature_cols = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "last_known_tvt", "typewell_corr_last300", "typewell_mae_last300",
    "typewell_best_lag_corr", "typewell_best_lag_tvt", "typewell_best_lag_mae", "target_rows", "target_tvt_span", "target_start_gap",
]
X_features = well_table[feature_cols].copy()
feature_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
X = feature_pipe.fit_transform(X_features)

kmeans = KMeans(n_clusters=CONFIG["n_descriptor_clusters"], random_state=CONFIG["random_state"], n_init=20)
well_table["descriptor_cluster"] = kmeans.fit_predict(X)

cluster_summary = (
    well_table.groupby("descriptor_cluster")
    .agg(n_wells=("wellname", "count"), n_rows=("total_rows", "sum"), mean_stress=("combined_stress_score", "mean"))
    .reset_index()
    .sort_values(["n_rows", "mean_stress"], ascending=[False, False])
)
cluster_to_fold = {}
fold_load = {fold: 0 for fold in range(1, CONFIG["n_folds"] + 1)}
for row in cluster_summary.itertuples(index=False):
    fold = min(fold_load, key=fold_load.get)
    cluster_to_fold[row.descriptor_cluster] = fold
    fold_load[fold] += row.n_rows
well_table["descriptor_cluster_fold"] = well_table["descriptor_cluster"].map(cluster_to_fold).astype(int)

well_table["analog_distance_fold"] = pd.qcut(well_table["selected_analog_distance"], CONFIG["n_folds"], labels=False, duplicates="drop") + 1
well_table["extrapolation_fold"] = pd.qcut(well_table["extrapolation_stress_score"], CONFIG["n_folds"], labels=False, duplicates="drop") + 1
well_table["stress_rank"] = well_table["combined_stress_score"].rank(method="first", ascending=False)

hybrid_fold = []
fold_rows = {fold: 0 for fold in range(1, CONFIG["n_folds"] + 1)}
fold_counts = {fold: 0 for fold in range(1, CONFIG["n_folds"] + 1)}
for row in well_table.sort_values(["combined_stress_score", "selected_analog_distance"], ascending=[False, False]).itertuples():
    fold = min(fold_rows, key=lambda f: (fold_rows[f], fold_counts[f]))
    hybrid_fold.append((row.wellname, fold))
    fold_rows[fold] += int(row.total_rows)
    fold_counts[fold] += 1
hybrid_map = dict(hybrid_fold)
well_table["strict_hybrid_fold"] = well_table["wellname"].map(hybrid_map).astype(int)

fold_cols = ["descriptor_cluster_fold", "analog_distance_fold", "extrapolation_fold", "strict_hybrid_fold"]
for col in fold_cols:
    well_table[col] = well_table[col].astype(int)

fold_assignments = well_table[[
    "wellname", "total_rows", "target_rows", "target_row_share", "target_tvt_span", "target_start_gap",
    "selected_analog_distance", "extrapolation_stress_score", "analog_stress_score", "combined_stress_score",
    "descriptor_cluster", *fold_cols,
]].copy()

fold_path = TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv"
fold_assignments.to_csv(fold_path, index=False)
shutil.copy2(fold_path, RUN_DIR / fold_path.name)

cluster_path = TABLE_DIR / "validation__strict_cv_descriptor_clusters__2026-06-26.csv"
cluster_summary.to_csv(cluster_path, index=False)
shutil.copy2(cluster_path, RUN_DIR / cluster_path.name)

fold_assignments.head()


,wellname,total_rows,target_rows,target_row_share,target_tvt_span,target_start_gap,selected_analog_distance,extrapolation_stress_score,analog_stress_score,combined_stress_score,descriptor_cluster,descriptor_cluster_fold,analog_distance_fold,extrapolation_fold,strict_hybrid_fold
0,000d7d20,5278,3836,0.726790,15.20,-21.29,0.316157,0.268111,0.241915,0.234039,1,1,1,1,2
1,00bbac68,7559,6014,0.795608,50.19,-35.23,0.416028,0.685640,0.561449,0.642173,3,1,2,3,1
2,00e12e8b,6384,4301,0.673716,29.59,-17.77,0.578991,0.379204,0.865459,0.665273,2,2,3,1,1
3,015fe0d2,5950,4296,0.722017,38.61,-8.27,0.728474,0.560964,0.957309,0.608190,4,3,3,2,3
4,01869cd4,6850,5557,0.811241,39.00,-17.19,0.803998,0.726876,0.978008,0.674054,6,2,3,3,2


## Fold Stress Diagnostics


In [4]:
def nearest_train_distance(fold_col):
    values = []
    for fold in sorted(well_table[fold_col].unique()):
        valid_idx = well_table[fold_col].eq(fold).to_numpy()
        train_idx = ~valid_idx
        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(X[train_idx])
        dists, _ = nn.kneighbors(X[valid_idx])
        tmp = well_table.loc[valid_idx, ["wellname"]].copy()
        tmp["fold_design"] = fold_col
        tmp["fold"] = int(fold)
        tmp["nearest_train_descriptor_distance"] = dists[:, 0]
        values.append(tmp)
    return pd.concat(values, ignore_index=True)

nearest_diag = pd.concat([nearest_train_distance(col) for col in fold_cols], ignore_index=True)
nearest_path = TABLE_DIR / "validation__strict_cv_nearest_train_distance__2026-06-26.csv"
nearest_diag.to_csv(nearest_path, index=False)
shutil.copy2(nearest_path, RUN_DIR / nearest_path.name)

summaries = []
for col in fold_cols:
    fold_summary = (
        well_table.groupby(col)
        .agg(
            n_wells=("wellname", "count"),
            n_rows=("total_rows", "sum"),
            target_rows=("target_rows", "sum"),
            mean_target_share=("target_row_share", "mean"),
            mean_target_tvt_span=("target_tvt_span", "mean"),
            mean_target_start_gap=("target_start_gap", "mean"),
            mean_selected_analog_distance=("selected_analog_distance", "mean"),
            mean_extrapolation_stress=("extrapolation_stress_score", "mean"),
            mean_combined_stress=("combined_stress_score", "mean"),
            mean_abs_bias_delta=("abs_bias_delta", "mean"),
        )
        .reset_index()
        .rename(columns={col: "fold"})
    )
    nn_summary = nearest_diag.loc[nearest_diag["fold_design"].eq(col)].groupby("fold")["nearest_train_descriptor_distance"].agg(["mean", "median", "max"]).reset_index()
    nn_summary = nn_summary.rename(columns={"mean": "mean_nearest_train_descriptor_distance", "median": "median_nearest_train_descriptor_distance", "max": "max_nearest_train_descriptor_distance"})
    fold_summary = fold_summary.merge(nn_summary, on="fold", how="left")
    fold_summary.insert(0, "fold_design", col)
    summaries.append(fold_summary)
fold_summary_all = pd.concat(summaries, ignore_index=True)

scores = []
for col in fold_cols:
    sub = fold_summary_all.loc[fold_summary_all["fold_design"].eq(col)].copy()
    row_imbalance = sub["n_rows"].max() / max(sub["n_rows"].min(), 1)
    well_imbalance = sub["n_wells"].max() / max(sub["n_wells"].min(), 1)
    stress_range = sub["mean_combined_stress"].max() - sub["mean_combined_stress"].min()
    analog_range = sub["mean_selected_analog_distance"].max() - sub["mean_selected_analog_distance"].min()
    nn_mean = sub["mean_nearest_train_descriptor_distance"].mean()
    nn_max = sub["max_nearest_train_descriptor_distance"].max()
    balance_penalty = (row_imbalance - 1.0) + 0.5 * (well_imbalance - 1.0)
    strictness_score = nn_mean + 0.5 * nn_max + 2.0 * stress_range + analog_range - balance_penalty
    scores.append({
        "fold_design": col,
        "row_imbalance": row_imbalance,
        "well_imbalance": well_imbalance,
        "stress_range": stress_range,
        "analog_distance_range": analog_range,
        "mean_nearest_train_descriptor_distance": nn_mean,
        "max_nearest_train_descriptor_distance": nn_max,
        "strictness_score": strictness_score,
    })
fold_scores = pd.DataFrame(scores).sort_values("strictness_score", ascending=False).reset_index(drop=True)
fold_scores["recommended_use"] = np.where(fold_scores["fold_design"].eq(CONFIG["recommended_fold_column"]), "default_model_cv", "diagnostic_stress_test")

summary_path = TABLE_DIR / "validation__strict_cv_fold_summary__2026-06-26.csv"
score_path = TABLE_DIR / "validation__strict_cv_design_scores__2026-06-26.csv"
fold_summary_all.to_csv(summary_path, index=False)
fold_scores.to_csv(score_path, index=False)
shutil.copy2(summary_path, RUN_DIR / summary_path.name)
shutil.copy2(score_path, RUN_DIR / score_path.name)

print(fold_scores.to_string(index=False))
fold_summary_all.head(12)


            fold_design  row_imbalance  well_imbalance  stress_range  analog_distance_range  mean_nearest_train_descriptor_distance  max_nearest_train_descriptor_distance  strictness_score        recommended_use
   analog_distance_fold       1.020798        1.003891      0.229472               0.324183                                2.560574                               8.009654          7.325784 diagnostic_stress_test
descriptor_cluster_fold       1.121338        1.088000      0.041398               0.017582                                2.871822                               7.705233          6.659478 diagnostic_stress_test
     extrapolation_fold       1.421818        1.003891      0.143834               0.016035                                2.613298                               6.903382          5.944928 diagnostic_stress_test
     strict_hybrid_fold       1.000058        1.023529      0.001379               0.017889                                2.435676                     

,fold_design,fold,n_wells,n_rows,target_rows,mean_target_share,mean_target_tvt_span,mean_target_start_gap,mean_selected_analog_distance,mean_extrapolation_stress,mean_combined_stress,mean_abs_bias_delta,mean_nearest_train_descriptor_distance,median_nearest_train_descriptor_distance,max_nearest_train_descriptor_distance
0,descriptor_cluster_fold,1,250,1768270,1361739,0.759734,28.811720,-17.840280,0.416678,0.570018,0.510800,-0.100967,2.847243,2.745550,6.266046
1,descriptor_cluster_fold,2,272,1747056,1256094,0.710397,29.175294,-18.917243,0.434260,0.452405,0.515661,-0.039946,2.885167,2.675879,7.705233
2,descriptor_cluster_fold,3,251,1576929,1166156,0.731362,30.252151,-20.786215,0.425678,0.483830,0.474263,-0.137088,2.883057,2.747531,7.261211
3,analog_distance_fold,1,258,1716234,1283816,0.738737,28.320853,-19.176667,0.277813,0.509120,0.385768,-0.004703,2.219145,2.183766,4.093055
4,analog_distance_fold,2,257,1694754,1258341,0.732570,29.831907,-19.677043,0.397443,0.498798,0.500934,-0.167145,2.436463,2.314515,5.546945
5,analog_distance_fold,3,258,1681267,1241832,0.728173,30.071008,-18.675659,0.601996,0.494015,0.615239,-0.102119,3.026113,2.857545,8.009654
6,extrapolation_fold,1,258,1401170,952868,0.673857,22.166279,-20.656279,0.436401,0.302320,0.423866,0.067409,2.648934,2.578927,6.738570
7,extrapolation_fold,2,257,1698876,1254321,0.736648,30.774786,-20.408366,0.420366,0.504169,0.510412,-0.193556,2.469111,2.374768,5.551986
8,extrapolation_fold,3,258,1992209,1576800,0.788991,35.286357,-16.467558,0.420574,0.695465,0.567700,-0.147922,2.721848,2.583289,6.903382
9,strict_hybrid_fold,1,261,1697452,1253637,0.729297,29.930038,-19.766935,0.435747,0.491741,0.500732,-0.052824,2.459840,2.375618,6.659501


## Stress Samples


In [5]:
stress_sample = well_table.sort_values(["combined_stress_score", "selected_analog_distance"], ascending=[False, False]).head(40)
stress_sample = stress_sample[[
    "wellname", "total_rows", "target_rows", "target_row_share", "target_tvt_span", "target_start_gap",
    "selected_analog_distance", "selected_analogs", "combined_stress_score", "descriptor_cluster",
    "descriptor_cluster_fold", "analog_distance_fold", "extrapolation_fold", "strict_hybrid_fold",
]]
sample_path = TABLE_DIR / "validation__strict_cv_stress_well_sample__2026-06-26.csv"
stress_sample.to_csv(sample_path, index=False)
shutil.copy2(sample_path, RUN_DIR / sample_path.name)
stress_sample.head(15)


,wellname,total_rows,target_rows,target_row_share,target_tvt_span,target_start_gap,selected_analog_distance,selected_analogs,combined_stress_score,descriptor_cluster,descriptor_cluster_fold,analog_distance_fold,extrapolation_fold,strict_hybrid_fold
506,a959858c,5770,4404,0.763258,90.53,0.53,1.064857,f6d009f4,0.886117,6,2,3,3,1
104,25050f63,7155,6124,0.855905,48.36,0.18,0.992466,eba6605e,0.845440,6,2,3,3,2
625,d1ecf309,7864,6168,0.784334,26.45,-10.09,0.707352,504c8b08,0.829730,8,1,3,3,3
624,d1457cc5,8443,6920,0.819614,26.27,-17.87,0.668985,7271dd80,0.829269,3,1,3,3,1
68,197f8a5a,8248,6310,0.765034,52.16,-5.24,0.532609,eb06a4e7,0.827911,2,2,3,3,2
743,f6d009f4,8003,6715,0.839060,84.48,0.28,0.409909,939d9c34,0.822372,6,2,2,3,3
92,201661b5,8136,5955,0.731932,45.97,-24.92,1.707816,b32e4903,0.820270,2,2,3,3,1
365,7ff89f8f,9539,7337,0.769158,40.97,-15.49,0.537436,27c3155b,0.815403,2,2,3,3,2
222,4a8ecc0b,7374,5809,0.787768,39.22,-20.07,0.785788,9c8375f7,0.810091,8,1,3,3,3
221,4a335117,6727,5015,0.745503,47.29,0.06,0.511717,bb682ebd,0.803477,6,2,3,3,1


## Report Export


In [6]:
recommended = fold_scores.loc[fold_scores["fold_design"].eq(CONFIG["recommended_fold_column"])].iloc[0]
strictest = fold_scores.iloc[0]
hybrid_summary = fold_summary_all.loc[fold_summary_all["fold_design"].eq(CONFIG["recommended_fold_column"])]

findings = pd.DataFrame([
    {"finding": "Default fold design", "value": CONFIG["recommended_fold_column"], "implication": "Use this fold column first for LightGBM/XGBoost strict CV."},
    {"finding": "Strictest diagnostic design", "value": strictest["fold_design"], "implication": "Use as an additional stress test when a model looks promising."},
    {"finding": "Hybrid row imbalance", "value": f"{recommended['row_imbalance']:.3f}", "implication": "Hybrid folds preserve usable training/validation size balance."},
    {"finding": "Hybrid nearest-train descriptor distance", "value": f"mean {recommended['mean_nearest_train_descriptor_distance']:.3f}, max {recommended['max_nearest_train_descriptor_distance']:.3f}", "implication": "Folds are stricter than random well splits but still feasible."},
    {"finding": "High-stress wells are explicit", "value": f"{len(stress_sample)} sampled wells exported", "implication": "Later model errors can be inspected against a stable stress set."},
])
findings_path = TABLE_DIR / "validation__strict_cv_findings__2026-06-26.csv"
findings.to_csv(findings_path, index=False)
shutil.copy2(findings_path, RUN_DIR / findings_path.name)

report = f"""# Strict CV Fold Design - 2026-06-26

## Summary

This notebook creates reusable well-level fold assignments that deliberately stress descriptor clusters, far analogs, and high-extrapolation wells. The recommended default for the next modeling notebooks is `{CONFIG['recommended_fold_column']}` because it balances row count while distributing high-stress wells across folds.

## Fold Designs

- `descriptor_cluster_fold`: holds out descriptor clusters built from known-zone geometry, GR summaries, trajectory span, and target-zone shape descriptors.
- `analog_distance_fold`: buckets wells by selected analog distance to stress weak analog transfer.
- `extrapolation_fold`: buckets wells by target-zone length/share/span/gap stress.
- `strict_hybrid_fold`: greedily balances rows while spreading combined analog and extrapolation stress.

## Recommendation

Use `strict_hybrid_fold` as the first strict-CV column for LightGBM/XGBoost. Treat `descriptor_cluster_fold`, `analog_distance_fold`, and `extrapolation_fold` as follow-up diagnostic stress tests for any candidate that beats the current Huber + Hilbert baseline locally.

## Key Diagnostics

- Hybrid row imbalance: **{recommended['row_imbalance']:.3f}**.
- Hybrid well imbalance: **{recommended['well_imbalance']:.3f}**.
- Hybrid mean nearest-training descriptor distance: **{recommended['mean_nearest_train_descriptor_distance']:.3f}**.
- Strictest diagnostic fold design by score: `{strictest['fold_design']}`.

## Next Steps

1. Re-score Huber + Hilbert and analog-gated variants with `strict_hybrid_fold` if feasible.
2. Start `02_trees/06_lightgbm_strict_cv.ipynb` using this fold assignment table.
3. Promote a boosted-tree notebook only if it improves strict CV by at least **0.10 RMSE** or explains the analog public/local mismatch.
"""
report_path = REPORT_DIR / "validation__strict_cv_fold_design__2026-06-26.md"
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(report)


# Strict CV Fold Design - 2026-06-26

## Summary

This notebook creates reusable well-level fold assignments that deliberately stress descriptor clusters, far analogs, and high-extrapolation wells. The recommended default for the next modeling notebooks is `strict_hybrid_fold` because it balances row count while distributing high-stress wells across folds.

## Fold Designs

- `descriptor_cluster_fold`: holds out descriptor clusters built from known-zone geometry, GR summaries, trajectory span, and target-zone shape descriptors.
- `analog_distance_fold`: buckets wells by selected analog distance to stress weak analog transfer.
- `extrapolation_fold`: buckets wells by target-zone length/share/span/gap stress.
- `strict_hybrid_fold`: greedily balances rows while spreading combined analog and extrapolation stress.

## Recommendation

Use `strict_hybrid_fold` as the first strict-CV column for LightGBM/XGBoost. Treat `descriptor_cluster_fold`, `analog_distance_fold`, and `extrapolation_fold`